# Tableau Data Preparation
## Repsol Capstone Project — Sprint 2

**Goal:** Produce a single, Tableau-optimised CSV that consolidates historical consumption,
test-set predictions, and the 24-month SARIMA forecast into one flat file.

**Inputs:**
- `data/processed/biodiesel_targets.csv` — historical 2023–2025
- `data/processed/predictions.csv` — test-set predictions (all models, 2025)
- `data/processed/forecast_24m.csv` — 24-month forecasts (2026–2027)
- `data/processed/metrics.csv` — MAE / RMSE / MAPE

**Outputs:**
- `data/processed/tableau_dashboard_data.csv` — main flat file (all series + metadata)
- `data/processed/tableau_metrics.csv` — model performance table
- `data/processed/tableau_forecast_pivot.csv` — SARIMA forecast pivoted (months as rows, regions as columns)

## 0. Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

NOTEBOOK_DIR = Path().resolve()
REPO_ROOT    = NOTEBOOK_DIR.parent
DATA         = REPO_ROOT / 'data' / 'processed'

df_hist    = pd.read_csv(DATA_INPUTS   / 'consumo_biodiesel_targets.csv')
df_pred    = pd.read_csv(DATA_OUTPUTS  / 'predicciones_test_2025.csv')
df_fc      = pd.read_csv(DATA_OUTPUTS  / 'forecast_24m_sarima_rf_xgb.csv')
df_metrics = pd.read_csv(DATA_OUTPUTS  / 'metricas_modelos.csv')

print('Historical shape:  ', df_hist.shape)
print('Predictions shape: ', df_pred.shape)
print('Forecast shape:    ', df_fc.shape)
print('Metrics shape:     ', df_metrics.shape)

Historical shape:   (180, 4)
Predictions shape:  (180, 5)
Forecast shape:     (360, 4)
Metrics shape:      (15, 5)


## 1. Build Master Flat File

### What?
Combine all series into one long-format table with consistent columns:
`Fecha_Date | Target | Model | Type | Valor_Tm | Anyo | Mes | Trimestre | Is_Primary`

### Why?
Tableau works best with a single long-format data source. Separate CSVs require
relationships or data blending — a single flat file keeps the workbook simple and fast.

### Type taxonomy
| Type | Description |
|------|-------------|
| `Historico` | Actual reported consumption 2023–2024 (training period) |
| `Historico_Test` | Actual reported consumption 2025 (test period — real, not modelled) |
| `Prediccion_Test` | Model predictions for 2025 test set |
| `Forecast` | Model forecasts 2026–2027 (future, not observed) |

In [2]:
rows = []

# ── 1. Historical actuals 2023–2024 (training period) ──────────────────────
hist_train = df_hist[df_hist['Fecha'] < '2025-01'].copy()
for _, r in hist_train.iterrows():
    rows.append({'Fecha': r['Fecha'], 'Target': r['Target'],
                 'Model': 'Actual', 'Type': 'Historico',
                 'Valor_Tm': r['Consumo_Tm'], 'Is_Primary': True})

# ── 2. Historical actuals 2025 (test period — observed truth) ───────────────
# Pull actuals from predictions file (same values, just under SARIMA rows)
actuals_2025 = df_pred[df_pred['Model'] == 'SARIMA'][['Fecha', 'Target', 'Actual']].drop_duplicates()
for _, r in actuals_2025.iterrows():
    rows.append({'Fecha': r['Fecha'], 'Target': r['Target'],
                 'Model': 'Actual', 'Type': 'Historico_Test',
                 'Valor_Tm': r['Actual'], 'Is_Primary': True})

# ── 3. Test-set predictions 2025 (all 3 models) ─────────────────────────────
for _, r in df_pred.iterrows():
    rows.append({'Fecha': r['Fecha'], 'Target': r['Target'],
                 'Model': r['Model'], 'Type': 'Prediccion_Test',
                 'Valor_Tm': r['Pred'],
                 'Is_Primary': r['Model'] == 'SARIMA'})

# ── 4. 24-month forecast 2026–2027 (all 3 models) ──────────────────────────
for _, r in df_fc.iterrows():
    rows.append({'Fecha': r['Fecha'], 'Target': r['Target'],
                 'Model': r['Model'], 'Type': 'Forecast',
                 'Valor_Tm': r['Forecast'],
                 'Is_Primary': r['Model'] == 'SARIMA'})

df_master = pd.DataFrame(rows)

# ── Date columns Tableau needs ──────────────────────────────────────────────
df_master['Fecha_Date'] = pd.to_datetime(df_master['Fecha'] + '-01')
df_master['Anyo']       = df_master['Fecha_Date'].dt.year
df_master['Mes']        = df_master['Fecha_Date'].dt.month
df_master['Trimestre']  = 'Q' + df_master['Fecha_Date'].dt.quarter.astype(str)
df_master['Fecha_Date'] = df_master['Fecha_Date'].dt.strftime('%Y-%m-%d')  # ISO string

# ── Column order ────────────────────────────────────────────────────────────
df_master = df_master[[
    'Fecha_Date', 'Fecha', 'Anyo', 'Mes', 'Trimestre',
    'Target', 'Model', 'Type', 'Valor_Tm', 'Is_Primary'
]].sort_values(['Target', 'Model', 'Fecha_Date']).reset_index(drop=True)

print(f'Master shape: {df_master.shape}')
print(f'\nType breakdown:')
print(df_master.groupby('Type')['Valor_Tm'].count())
print(f'\nModel breakdown:')
print(df_master.groupby('Model')['Valor_Tm'].count())
df_master.head(10)

Master shape: (720, 10)

Type breakdown:
Type
Forecast           360
Historico          120
Historico_Test      60
Prediccion_Test    180
Name: Valor_Tm, dtype: int64

Model breakdown:
Model
Actual           180
Random Forest    180
SARIMA           180
XGBoost          180
Name: Valor_Tm, dtype: int64


,Fecha_Date,Fecha,Anyo,Mes,Trimestre,Target,Model,Type,Valor_Tm,Is_Primary
0,2023-01-01,2023-01,2023,1,Q1,Andalucía,Actual,Historico,0.0,True
1,2023-02-01,2023-02,2023,2,Q1,Andalucía,Actual,Historico,0.0,True
2,2023-03-01,2023-03,2023,3,Q1,Andalucía,Actual,Historico,0.0,True
3,2023-04-01,2023-04,2023,4,Q2,Andalucía,Actual,Historico,46.0,True
4,2023-05-01,2023-05,2023,5,Q2,Andalucía,Actual,Historico,0.0,True
5,2023-06-01,2023-06,2023,6,Q2,Andalucía,Actual,Historico,0.0,True
6,2023-07-01,2023-07,2023,7,Q3,Andalucía,Actual,Historico,0.0,True
7,2023-08-01,2023-08,2023,8,Q3,Andalucía,Actual,Historico,0.0,True
8,2023-09-01,2023-09,2023,9,Q3,Andalucía,Actual,Historico,19.0,True
9,2023-10-01,2023-10,2023,10,Q4,Andalucía,Actual,Historico,43.0,True


## 2. Metrics Table

Add a display-ready model ranking column.

In [3]:
df_met = df_metrics.copy()

# Rank each model within each target (1 = best MAPE)
df_met['Rank'] = df_met.groupby('Target')['MAPE'].rank(method='min').astype(int)
df_met['Mejor_Modelo'] = df_met['Rank'] == 1
df_met['MAPE_Label']   = df_met['MAPE'].apply(lambda x: f'{x:.1f}%')
df_met['MAE_Label']    = df_met['MAE'].apply(lambda x: f'{x:,.0f} Tm')

print(df_met.sort_values(['Target','Rank']).to_string(index=False))

   Target         Model     MAE    RMSE  MAPE  Rank  Mejor_Modelo MAPE_Label MAE_Label
Andalucía        SARIMA   897.9  1077.0  52.5     1          True      52.5%    898 Tm
Andalucía Random Forest   977.8  1157.9  56.7     2         False      56.7%    978 Tm
Andalucía       XGBoost  1050.5  1196.7  65.6     3         False      65.6%  1,050 Tm
 Cataluña        SARIMA  1428.6  1613.8  47.2     1          True      47.2%  1,429 Tm
 Cataluña       XGBoost  1720.3  1813.8  58.1     2         False      58.1%  1,720 Tm
 Cataluña Random Forest  1789.5  1893.3  60.3     3         False      60.3%  1,790 Tm
   Madrid       XGBoost  1697.9  1815.9  60.3     1          True      60.3%  1,698 Tm
   Madrid Random Forest  1844.8  1956.9  66.0     2         False      66.0%  1,845 Tm
   Madrid        SARIMA  9097.8 11968.6 318.6     3         False     318.6%  9,098 Tm
 Nacional        SARIMA  4277.6  4921.4  29.0     1          True      29.0%  4,278 Tm
 Nacional Random Forest 11221.7 12173.6  58

## 3. SARIMA Forecast Pivot (for summary table view)

In [4]:
TARGETS = ['Nacional', 'Madrid', 'Cataluña', 'Andalucía', 'Valencia']

sarima_fc = df_fc[df_fc['Model'] == 'SARIMA'].copy()
sarima_fc['Fecha_Date'] = pd.to_datetime(sarima_fc['Fecha'] + '-01').dt.strftime('%Y-%m-%d')
sarima_fc['Anyo']       = pd.to_datetime(sarima_fc['Fecha'] + '-01').dt.year
sarima_fc['Mes']        = pd.to_datetime(sarima_fc['Fecha'] + '-01').dt.month
sarima_fc['Trimestre']  = 'Q' + pd.to_datetime(sarima_fc['Fecha'] + '-01').dt.quarter.astype(str)

# Wide pivot for reading in Tableau / Excel
pivot = sarima_fc.pivot(index=['Fecha_Date','Anyo','Mes','Trimestre'],
                        columns='Target', values='Forecast').reset_index()
pivot.columns.name = None
# Reorder columns
pivot = pivot[['Fecha_Date','Anyo','Mes','Trimestre'] + [t for t in TARGETS if t in pivot.columns]]
pivot = pivot.sort_values('Fecha_Date').reset_index(drop=True)

print('SARIMA Forecast Pivot (2026–2027):')
print(pivot.to_string(index=False))

SARIMA Forecast Pivot (2026–2027):
Fecha_Date  Anyo  Mes Trimestre  Nacional  Madrid  Cataluña  Andalucía  Valencia
2026-01-01  2026    1        Q1   23556.5  3106.4    3497.8     2122.2    1286.6
2026-02-01  2026    2        Q1   24152.2  3234.6    3628.5     2236.0    1279.8
2026-03-01  2026    3        Q1   24910.6  3361.2    3754.7     2221.1    1275.3
2026-04-01  2026    4        Q2   25466.8  3478.4    3876.1     2224.0    1258.4
2026-05-01  2026    5        Q2   26328.3  3588.2    3987.9     2167.2    1239.4
2026-06-01  2026    6        Q2   26946.1  3691.7    4088.8     2135.2    1225.7
2026-07-01  2026    7        Q3   27369.3  3780.5    4181.6     2112.8    1201.9
2026-08-01  2026    8        Q3   26596.6  3848.3    4254.5     2157.3    1232.5
2026-09-01  2026    9        Q3   26463.4  3923.4    4324.4     2172.0    1236.8
2026-10-01  2026   10        Q4   26641.6  3987.7    4406.3     2170.3    1232.4
2026-11-01  2026   11        Q4   26657.4  4055.5    4465.2     2175.7    

## 4. Save Files

In [5]:
out1 = DATA_OUTPUTS  / 'tableau_dashboard.csv'
out2 = DATA_OUTPUTS  / 'tableau_metricas.csv'
out3 = DATA_OUTPUTS  / 'tableau_forecast_pivot.csv'

df_master.to_csv(out1, index=False, encoding='utf-8')
df_met.to_csv(out2, index=False, encoding='utf-8')
pivot.to_csv(out3, index=False, encoding='utf-8')

for p in [out1, out2, out3]:
    df = pd.read_csv(p)
    print(f'{p.name:<40} {df.shape[0]:>4} rows × {df.shape[1]:>2} cols')

tableau_dashboard_data.csv                720 rows × 10 cols
tableau_metrics.csv                        15 rows ×  9 cols
tableau_forecast_pivot.csv                 24 rows ×  9 cols


---
# Tableau Dashboard Guide

## Data source setup

Connect **3 data sources** in Tableau Desktop:

| # | File | Purpose |
|---|------|---------|
| 1 | `tableau_dashboard_data.csv` | Main data source (all views except metrics) |
| 2 | `tableau_metrics.csv` | Model performance table |
| 3 | `tableau_forecast_pivot.csv` | SARIMA forecast summary table |

**Field types to set manually after connecting:**

In `tableau_dashboard_data.csv`:
- `Fecha_Date` → **Date** (format: YYYY-MM-DD)
- `Anyo` → **Discrete dimension** (Integer)
- `Mes` → **Discrete dimension** (Integer)
- `Valor_Tm` → **Measure** (Number, decimal)
- `Is_Primary` → **Dimension** (Boolean)

---

## View 1 — Historical + Forecast Line Chart

**Purpose:** Main story — show explosive growth 2023–2025 continuing into 2026–2027.

**Steps:**
1. New sheet → rename **"Serie Completa"**
2. Columns: `Fecha_Date` (continuous, **Month** granularity)
3. Rows: `SUM(Valor_Tm)`
4. Filters:
   - `Type` → include **Historico**, **Historico_Test**, **Forecast**
   - `Model` → include **Actual**, **SARIMA**
   - `Target` → add to **Filter shelf** (show filter as single-value dropdown)
5. Color: `Type`
   - Historico → `#555555` (grey)
   - Historico_Test → `#000000` (black)
   - Forecast → `#2C7BB6` (blue)
6. Mark type → **Line**
7. Add **Reference Line**: at `2026-01-01`, label "Inicio Forecast", grey dashed
   *(Analytics pane → Reference Line → Table → Constant → 2026-01-01)*
8. Dual axis trick for the forecast confidence band:
   - Duplicate `SUM(Valor_Tm)` onto rows → dual axis
   - Change second axis mark type to **Area**
   - Set opacity to 15%
   - This requires adding `Valor_Tm * 0.8` and `Valor_Tm * 1.2` as calculated fields (see Calculated Fields section below)

---

## View 2 — Regional Comparison (Small Multiples)

**Purpose:** Compare all 5 regions side-by-side — spot which region is driving national growth.

**Steps:**
1. New sheet → rename **"Comparativa Regional"**
2. Columns: `Fecha_Date` (Month)
3. Rows: `SUM(Valor_Tm)`
4. Color: `Target`
   - Nacional → `#FF6B35`, Madrid → `#004E89`, Cataluña → `#1A936F`
   - Andalucía → `#C84B31`, Valencia → `#8E44AD`
5. Filters: `Type` = Historico + Historico_Test + Forecast; `Model` = Actual + SARIMA
6. To make small multiples: drag `Target` to **Rows** (between the row shelf and `SUM(Valor_Tm)`)
7. Check **"Show Headers"** → unchecked for the Target axis
8. Label each panel: right-click Target in Rows → **Annotate → Mark** (or use a title)

---

## View 3 — Model Performance Metrics Table

**Purpose:** Show which model won for each region and what the error was.

**Data source:** `tableau_metrics.csv`

**Steps:**
1. New sheet → rename **"Métricas de Modelos"**
2. Rows: `Target`
3. Columns: `Model`
4. Marks: **Text** → drag `MAPE_Label` to Text, `MAE_Label` to Tooltip
5. Color: `Rank`
   - Use **Stepped color** (3 steps): green (Rank 1) → yellow (2) → red (3)
   - Drag `Rank` to Color mark
6. Add `Mejor_Modelo` to **Shape** (checkmark for TRUE)
7. Sort rows: Nacional first (it's the KPI), then alphabetical

---

## View 4 — Monthly Forecast Table (2026–2027)

**Data source:** `tableau_forecast_pivot.csv`

**Steps:**
1. New sheet → rename **"Tabla Forecast Mensual"**
2. Rows: `Anyo`, `Trimestre`, `Mes`
3. Columns: `Nacional`, `Madrid`, `Cataluña`, `Andalucía`, `Valencia` (measure values)
4. Marks: **Text** → format as `#,##0 Tm`
5. Add subtotals: **Analysis → Totals → Show Row Grand Totals** (gives annual total per region)
6. Conditional formatting: color scale on each column (white → blue, proportional to value)

---

## View 5 — Model Comparison on Test Set (2025)

**Purpose:** Validate the models — show actual 2025 vs all 3 predictions.

**Steps:**
1. New sheet → rename **"Validación 2025"**
2. Columns: `Fecha_Date` (Month)
3. Rows: `SUM(Valor_Tm)`
4. Color: `Model`
   - Actual → `#000000`, SARIMA → `#2C7BB6`, Random Forest → `#1A936F`, XGBoost → `#D7191C`
5. Filter: `Type` = **Historico_Test** + **Prediccion_Test** (this limits to 2025 only)
6. Filter: `Target` (dropdown)
7. Mark the Actual line thicker: right-click Actual in Color legend → **Edit** → Size 3px

---

## Calculated Fields (copy-paste into Tableau)

```
// Is Forecast period?
[Is_Forecast]
= [Type] = "Forecast"

// Forecast lower bound (±20%)
[Forecast_Low]
= IF [Type] = "Forecast" THEN [Valor_Tm] * 0.80 ELSE NULL END

// Forecast upper bound (±20%)
[Forecast_High]
= IF [Type] = "Forecast" THEN [Valor_Tm] * 1.20 ELSE NULL END

// YoY growth % (requires LOOKUP)
[YoY_Growth_%]
= (ZN(SUM([Valor_Tm])) - LOOKUP(ZN(SUM([Valor_Tm])), -12)) 
  / ABS(LOOKUP(ZN(SUM([Valor_Tm])), -12)) * 100

// Display label (clean type names in Spanish)
[Tipo_Label]
= CASE [Type]
    WHEN "Historico"       THEN "Histórico (2023–2024)"
    WHEN "Historico_Test"  THEN "Real 2025"
    WHEN "Prediccion_Test" THEN "Predicción Test"
    WHEN "Forecast"        THEN "Forecast 2026–2027"
  END
```

---

## Dashboard Layout

**Recommended layout (1400×900 px, Floating):**

```
┌──────────────────────────────────────────────────────────────┐
│  TÍTULO: Repsol Diesel Nexa — Previsión de Demanda 2026–2027 │
│  Filtros globales: [Target ▾]  [Año ▾]                        │
├──────────────────────────────────────────────────┬───────────┤
│                                                  │  Métricas │
│  View 1: Serie Completa (large, ~60% width)      │  Modelos  │
│                                                  │  (View 3) │
├────────────────────────┬─────────────────────────┴───────────┤
│  View 2: Comparativa   │  View 5: Validación 2025            │
│  Regional              │                                     │
├────────────────────────┴─────────────────────────────────────┤
│  View 4: Tabla Forecast Mensual SARIMA (full width)          │
└──────────────────────────────────────────────────────────────┘
```

**Global filter setup:**
- Click the `Target` filter on View 1 → **Apply to Worksheets → All Using This Data Source**
- This syncs the dropdown across all views automatically

---

## Sprint 2 — Tableau Prep: DONE ✅

| File | Use |
|------|-----|
| `tableau_dashboard_data.csv` | Views 1, 2, 5 |
| `tableau_metrics.csv` | View 3 |
| `tableau_forecast_pivot.csv` | View 4 |